# Terrazas y knickpoints en perfiles fluviales

Visualizamos las terrazas junto al cauce y delimitamos dos intervalos:

1. Tramo costero para evaluar la calibración de K.
2. Tramo entre la cota de la terraza y el knickpoint.

Las asociaciones geométricas no validan por sí solas la cronología
ni la transferencia de tasas de levantamiento al río.

In [ ]:
from pathlib import Path
import json
import re
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.integrate import cumulative_trapezoid
from scipy.stats import linregress
from IPython.display import display

# ── ENTRADAS ──────────────────────────────────────────────────────
MEMORIA = Path(
    r"C:\datos\Memoria"
)
PRODUCTOS = MEMORIA / "Codigos Stanford" / "Productos"

CARPETA_PERFILES = PRODUCTOS / "info_perfiles" / "20260929"

ARCHIVO_KP = (
    PRODUCTOS / "analisis_knickpoints"
    / "20260924_194010_921014" / "catalogo_candidatos.csv"
)

ARCHIVO_TERRAZAS = MEMORIA / "Terrazas_revisadas_UTM18S.xlsx"
ARCHIVO_REVISION = Path(
    r"C:\datos\Memoria\Revision_muestras_Jara_2015.xlsx"
)

manifest = json.loads(
    (CARPETA_PERFILES / "manifest.json").read_text(encoding="utf-8")
)

if (
    manifest["unidad_distancia"] != "m"
    or manifest["unidad_area"] != "m2"
):
    raise ValueError("Se requieren distancias en m y áreas en m².")

perfiles = {}
nombres = {}

for river_id, datos in manifest["rios"].items():
    p = pd.read_csv(CARPETA_PERFILES / datos["archivo"])
    columnas = ["distancia_m", "elevacion_m", "area_m2", "x_utm", "y_utm"]

    if not set(columnas).issubset(p.columns):
        raise ValueError(f"Faltan columnas en el río {river_id}.")

    valores = p[columnas].to_numpy(float)

    if not np.isfinite(valores).all():
        raise ValueError(
            f"Río {river_id}: hay datos no finitos. "
            "Revisar los huecos antes de proyectar o integrar."
        )

    if (
        np.any(np.diff(p["distancia_m"]) <= 0)
        or np.any(p["area_m2"] <= 0)
    ):
        raise ValueError(f"Río {river_id}: revisar distancias y áreas.")

    perfiles[str(river_id)] = p
    nombres[str(river_id)] = datos["nombre"]

knickpoints = pd.read_csv(ARCHIVO_KP, dtype={"ID": str})

if knickpoints["id_kp"].duplicated().any():
    raise ValueError("El catálogo contiene IDs de knickpoints duplicados.")

# Comprobar que el catálogo corresponde a estos mismos perfiles.
for _, k in knickpoints.iterrows():
    rid = k["ID"]

    if rid not in perfiles:
        raise ValueError(f"El catálogo contiene un río no cargado: {rid}")

    p = perfiles[rid]
    j = int(k["indice"])

    if not 0 <= j < len(p):
        raise ValueError(f"Índice incompatible: {k['id_kp']}")

    columnas = ["distancia_m", "elevacion_m", "x_utm", "y_utm"]

    if not np.allclose(
        p.iloc[j][columnas].to_numpy(float),
        k[columnas].to_numpy(float),
        atol=0.1, rtol=0,
    ):
        raise ValueError(
            f"{k['id_kp']}: catálogo y perfil no coinciden. "
            "Usa la exportación del mismo análisis."
        )

terrazas_originales = pd.read_excel(
    ARCHIVO_TERRAZAS, sheet_name="Hoja1"
)
revision_muestras = pd.read_excel(
    ARCHIVO_REVISION, sheet_name="Resumen"
)

print(f"Perfiles: {len(perfiles)}")
print(f"Candidatos a knickpoint: {len(knickpoints)}")
print(f"Registros en la planilla de terrazas: {len(terrazas_originales)}")

## Asociar terrazas y perfiles

La correspondencia se realiza por nombre del río.
Conservamos las elevaciones, incertidumbres y observaciones originales.

La proyección sobre el cauce sirve para visualizar la posición
longitudinal de cada terraza; no establece continuidad geomorfológica.

In [ ]:
def normalizar_nombre(texto):
    texto = unicodedata.normalize("NFKD", str(texto))
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    texto = texto.lower().strip()
    texto = re.sub(r"^s/n\s*", "", texto)
    texto = re.sub(r"^(rio|estero)\s+", "", texto)
    texto = re.sub(r"\s+", " ", texto)

    # Diferencias de escritura presentes en tus archivos.
    alias = {
        "buchupero": "buchupureo",
        "cobequecura": "cobquecura",
    }
    return alias.get(texto, texto)


nombre_a_id = {
    normalizar_nombre(nombre): rid
    for rid, nombre in nombres.items()
}

if len(nombre_a_id) != len(nombres):
    raise ValueError("Hay nombres de río ambiguos después de normalizar.")

t = terrazas_originales.copy()
t["fila_excel"] = np.arange(len(t)) + 2

# Estas columnas contienen filas agrupadas en la planilla.
t["Cauce"] = t["Cauce"].ffill()
t["cauce_excel"] = t["Cauce asociado"].fillna(t["Cauce"])

t["ID"] = t["cauce_excel"].map(
    lambda nombre: nombre_a_id.get(normalizar_nombre(nombre))
)

t["terraza_id"] = (
    t["Terraza"].astype("string")
    .str.strip().str.replace("-", "_", regex=False)
)

for columna in [
    "Elevación (m)", "Error elevación (m)",
    "Este UTM18S (m)", "Norte UTM18S (m)",
]:
    t[columna] = pd.to_numeric(t[columna], errors="coerce")

tiene_datos = np.isfinite(t[[
    "Elevación (m)", "Este UTM18S (m)", "Norte UTM18S (m)"
]].to_numpy(float)).all(axis=1)

terrazas_no_asociadas = t[
    ~tiene_datos | t["ID"].isna()
].copy()

terrazas = t[
    tiene_datos & t["ID"].notna()
].copy().reset_index(drop=True)

if not terrazas["CRS coordenadas UTM"].astype(str).str.contains(
    "32718", regex=False
).all():
    raise ValueError("Revisa el CRS de las terrazas; se espera EPSG:32718.")

# Una terraza puede estar propuesta para más de un río.
terrazas["asociacion_id"] = (
    "R" + terrazas["ID"].str.zfill(2)
    + "_" + terrazas["terraza_id"]
)

if terrazas["asociacion_id"].duplicated().any():
    raise ValueError("Hay asociaciones río–terraza duplicadas.")


def proyectar_terraza(p, este, norte):
    xy = p[["x_utm", "y_utm"]].to_numpy(float)
    inicio = xy[:-1]
    vector = xy[1:] - inicio
    longitud2 = np.sum(vector**2, axis=1)

    if np.any(longitud2 <= 0):
        raise ValueError("El cauce contiene segmentos espaciales de longitud cero.")

    punto = np.array([este, norte], dtype=float)
    fraccion = np.clip(
        np.sum((punto - inicio) * vector, axis=1) / longitud2,
        0, 1,
    )

    proyecciones = inicio + fraccion[:, None] * vector
    distancias = np.linalg.norm(proyecciones - punto, axis=1)
    j = int(np.argmin(distancias))
    f = fraccion[j]

    distancia = p["distancia_m"].to_numpy(float)
    elevacion = p["elevacion_m"].to_numpy(float)

    return {
        "distancia_proyectada_m": distancia[j] + f * (
            distancia[j + 1] - distancia[j]
        ),
        "distancia_lateral_m": distancias[j],
        "x_proyectado": proyecciones[j, 0],
        "y_proyectado": proyecciones[j, 1],
        "z_cauce_proyectado_m": elevacion[j] + f * (
            elevacion[j + 1] - elevacion[j]
        ),
    }


proyecciones = [
    proyectar_terraza(
        perfiles[fila["ID"]],
        fila["Este UTM18S (m)"],
        fila["Norte UTM18S (m)"],
    )
    for _, fila in terrazas.iterrows()
]

terrazas = pd.concat(
    [terrazas, pd.DataFrame(proyecciones)], axis=1
)

#display(terrazas[[
 #   "asociacion_id", "cauce_excel", "Elevación (m)",
  #  "Error elevación (m)", "distancia_lateral_m",
   # "Edad local o correlacionada",
#]])

print(
    "Registros sin asociación o sin coordenadas:",
    len(terrazas_no_asociadas),
)

## Visualización

Mostramos las terrazas y los candidatos a knickpoints sobre los
perfiles longitudinales y $\chi$–elevación.

Cada terraza conserva su elevación original y se ubica según su
proyección espacial sobre el cauce.

Esta visualización no define automáticamente los límites de análisis.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import cumulative_trapezoid

# ── MODIFICAR AQUÍ ────────────────────────────────────────────────
#RIOS_VER = ["1"]
# RIOS_VER = ["3", "4", "5", "6"]
RIOS_VER = list(perfiles)

THETA_VISUAL = 0.45
A0_VISUAL_M2 = 1_000_000.0

ETIQUETAR_KP = False
ETIQUETAR_TERRAZAS = True
MOSTRAR_COTAS_TERRAZAS = True
MOSTRAR_TABLA_KP = False

COLORES_KP = {
    "Local": "#D55E00",
    "TopoToolbox": "#7B2CBF",
    "Gailleton": "#009E73",
}


def mostrar(vista="ambos", rios=None):
    vista = vista.lower().strip()

    if vista not in ("chi", "longitudinal", "ambos"):
        raise ValueError('Usa "chi", "longitudinal" o "ambos".')

    if (
        not np.isfinite(THETA_VISUAL)
        or not np.isfinite(A0_VISUAL_M2)
        or A0_VISUAL_M2 <= 0
    ):
        raise ValueError("Revisa THETA_VISUAL y A0_VISUAL_M2.")

    tipos = (
        ["longitudinal", "chi"]
        if vista == "ambos"
        else [vista]
    )

    if rios is None:
        rios = list(perfiles)
    elif isinstance(rios, (str, int)):
        rios = [rios]

    for rid in map(str, rios):
        if rid not in perfiles:
            raise ValueError(f"No está cargado el río {rid}.")

        p = perfiles[rid]
        ts = terrazas[terrazas["ID"].astype(str).eq(rid)]
        ks = knickpoints[knickpoints["ID"].astype(str).eq(rid)]

        distancia = p["distancia_m"].to_numpy(float)
        elevacion = p["elevacion_m"].to_numpy(float)
        area = p["area_m2"].to_numpy(float)

        if (
            not np.isfinite(distancia).all()
            or not np.isfinite(area).all()
            or np.any(np.diff(distancia) <= 0)
            or np.any(area <= 0)
        ):
            raise ValueError(f"Río {rid}: revisar distancias y áreas.")

        # χ desde el inicio del perfil, sin modificar los datos originales.
        chi = cumulative_trapezoid(
            (A0_VISUAL_M2 / area) ** THETA_VISUAL,
            x=distancia,
            initial=0.0,
        )

        def coordenada_horizontal(posiciones, tipo):
            posiciones = np.asarray(posiciones, dtype=float)

            if (
                not np.isfinite(posiciones).all()
                or np.any(posiciones < distancia[0])
                or np.any(posiciones > distancia[-1])
            ):
                raise ValueError(
                    f"Río {rid}: hay posiciones fuera del perfil."
                )

            if tipo == "longitudinal":
                return posiciones / 1000

            return np.interp(posiciones, distancia, chi)

        fig, axs = plt.subplots(
            1,
            len(tipos),
            figsize=(15, 5) if len(tipos) == 2 else (10, 5),
            squeeze=False,
        )
        axs = axs.ravel()

        for ax, tipo in zip(axs, tipos):
            x_cauce = distancia / 1000 if tipo == "longitudinal" else chi

            ax.plot(
                x_cauce,
                elevacion,
                color="steelblue",
                linewidth=1.3,
                label="Cauce",
            )

            # ── Terrazas ──────────────────────────────────────────
            for numero, (_, terraza) in enumerate(ts.iterrows()):
                color = plt.get_cmap("tab10")(numero % 10)
                z = float(terraza["Elevación (m)"])
                error = terraza["Error elevación (m)"]

                x = float(coordenada_horizontal(
                    terraza["distancia_proyectada_m"],
                    tipo,
                ))

                ax.errorbar(
                    x,
                    z,
                    yerr=(
                        error
                        if np.isfinite(error) and error >= 0
                        else None
                    ),
                    fmt="s",
                    color=color,
                    markersize=6,
                    capsize=4,
                    zorder=5,
                    label=terraza["terraza_id"],
                )

                if MOSTRAR_COTAS_TERRAZAS:
                    ax.axhline(
                        z,
                        color=color,
                        linestyle=":",
                        alpha=0.45,
                    )

                if ETIQUETAR_TERRAZAS:
                    ax.annotate(
                        terraza["terraza_id"],
                        (x, z),
                        xytext=(5, 7),
                        textcoords="offset points",
                        fontsize=8,
                        color=color,
                    )

            # ── Knickpoints por método ────────────────────────────
            for metodo, grupo in ks.groupby("metodo"):
                color = COLORES_KP.get(metodo, "black")

                x_kp = coordenada_horizontal(
                    grupo["distancia_m"].to_numpy(float),
                    tipo,
                )

                ax.scatter(
                    x_kp,
                    grupo["elevacion_m"],
                    s=35,
                    color=color,
                    zorder=4,
                    label=metodo,
                )

                if ETIQUETAR_KP:
                    for x, (_, k) in zip(x_kp, grupo.iterrows()):
                        ax.annotate(
                            k["id_kp"],
                            (x, k["elevacion_m"]),
                            xytext=(4, 6),
                            textcoords="offset points",
                            fontsize=7,
                            color=color,
                        )

            if tipo == "chi":
                ax.set(
                    xlabel=r"$\chi$ (m)",
                    title=(
                        rf"Perfil $\chi$–elevación | "
                        rf"$\theta={THETA_VISUAL:g}$"
                    ),
                )
            else:
                ax.set(
                    xlabel="Distancia desde el outlet (km)",
                    title="Perfil longitudinal",
                )

            ax.set_ylabel("Elevación (m)")
            ax.grid(alpha=0.25)
            ax.legend(fontsize=8)

        fig.suptitle(
            f"{rid} — {nombres[rid]} | "
            "Terrazas proyectadas y candidatos"
        )
        fig.tight_layout()
        plt.show()
        plt.close(fig)

        if MOSTRAR_TABLA_KP:
            display(ks[[
                "id_kp",
                "metodo",
                "distancia_m",
                "elevacion_m",
                "me_interesa",
            ]])

# MODIFICAR AQUÍ
RIO_ID = list(perfiles)

display(
    terrazas.loc[
        terrazas["ID"].astype(str).eq(str(RIO_ID)),
        [
            "terraza_id",
            "Elevación (m)",
            "Error elevación (m)",
            "distancia_lateral_m",
            "Edad local o correlacionada",
            "Muestra(s) de respaldo de la correlación",
            "Alcance y observaciones de la asociación",
        ],
    ].sort_values("Elevación (m)", ascending=False)
)

# ── ELEGIR QUÉ MOSTRAR: deja activa una sola línea ─────────────────
#mostrar("chi", "6")                       # χ del río 6
# mostrar("longitudinal", "6")            # Longitudinal del río 6
mostrar("ambos", RIO_ID )                   # Ambos del río 6
# mostrar("chi", ["1", "3", "6"])          # χ de varios ríos
# mostrar("ambos", ["3", "6"])            # Ambos de varios ríos
# mostrar("ambos")                       # Todos los ríos

## Selección del corte

Elegimos una terraza y revisamos todas las posiciones donde
el perfil cruza su elevación.

El cruce por cota es distinto de la proyección espacial.
La selección permanece provisional hasta revisar su respaldo.

In [ ]:
# ── MODIFICAR AQUÍ ────────────────────────────────────────────────
RIO_ID = "6"
TERRAZA_ID = "JM2015_043"

seleccion = terrazas[
    terrazas["ID"].eq(RIO_ID)
    & terrazas["terraza_id"].eq(TERRAZA_ID)
]

if len(seleccion) != 1:
    raise ValueError("Selecciona una asociación río–terraza existente y única.")

terraza = seleccion.iloc[0]
p = perfiles[RIO_ID]

display(seleccion[[
    "asociacion_id", "Elevación (m)", "Error elevación (m)",
    "distancia_lateral_m",
    "Muestra(s) de respaldo de la correlación",
    "Edad local o correlacionada",
    "Alcance y observaciones de la asociación",
]])

# Mostrar la revisión documental de muestras mencionadas explícitamente.
respaldo = str(terraza["Muestra(s) de respaldo de la correlación"])
muestras_relacionadas = revision_muestras[
    revision_muestras["Sample ID"].astype(str).map(
        lambda muestra: muestra in respaldo
    )
]

display(muestras_relacionadas[[
    "Sample ID", "Edad medida tabla 1 (ka)",
    "Edad asignada (ka)", "U corregido (m/ka)",
    "Pendientes y evaluación", "Mi revisión manual",
]])


def cruces_por_cota(p, cota):
    x = p["distancia_m"].to_numpy(float)
    z = p["elevacion_m"].to_numpy(float)
    resultados = []

    for j in range(len(x) - 1):
        if z[j] == z[j + 1]:
            if z[j] == cota:
                raise ValueError(
                    "La cota coincide con un segmento plano. "
                    "Define y documenta un límite manual."
                )
            continue

        if min(z[j], z[j + 1]) <= cota <= max(z[j], z[j + 1]):
            fraccion = (cota - z[j]) / (z[j + 1] - z[j])
            resultados.append(x[j] + fraccion * (x[j + 1] - x[j]))

    return np.unique(np.round(resultados, 6))


cota = float(terraza["Elevación (m)"])
posiciones = cruces_por_cota(p, cota)

cruces = pd.DataFrame({
    "cruce_id": np.arange(1, len(posiciones) + 1),
    "distancia_m": posiciones,
    "distancia_km": posiciones / 1000,
})

display(cruces)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(p["distancia_m"] / 1000, p["elevacion_m"], color="steelblue")
ax.axhline(cota, color="darkorange", ls="--", label=TERRAZA_ID)

for _, cruce in cruces.iterrows():
    ax.scatter(cruce["distancia_km"], cota, color="black")
    ax.annotate(
        f"Cruce {int(cruce['cruce_id'])}",
        (cruce["distancia_km"], cota),
        xytext=(5, 8), textcoords="offset points",
    )

ax.set(xlabel="Distancia desde el outlet (km)", ylabel="Elevación (m)")
ax.grid(alpha=0.25)
ax.legend()
plt.show()
plt.close(fig)

# Mostrar cómo cambia la posición del corte con la incertidumbre vertical.
error = terraza["Error elevación (m)"]

if np.isfinite(error) and error >= 0:
    for etiqueta, altura in [
        ("Cota inferior", cota - error),
        ("Cota central", cota),
        ("Cota superior", cota + error),
    ]:
        print(
            etiqueta, f"({altura:.2f} m):",
            (cruces_por_cota(p, altura) / 1000).round(3),
            "km",
        )

## Tramos y sensibilidad a la concavidad

Definimos el final de la planicie aluvial, el cruce elegido
y el knickpoint.

Evaluamos qué concavidad hace más lineal cada tramo en χ–elevación.
Este resultado es una estimación geométrica, no una demostración
de equilibrio ni una concavidad definitiva.

In [ ]:
# ── COMPLETAR AQUÍ ────────────────────────────────────────────────
INICIO_ROCA_KM = 2.5 # Distancia donde termina la planicie aluvial.
CRUCE_ID = 2        # Número mostrado en la tabla anterior.
KNICKPOINT_ID = "R06_T_I00452"   # Copiar un id_kp del catálogo.

# Ejemplo de sintaxis, reemplazar por valores revisados:
# INICIO_ROCA_KM = 2.5
# CRUCE_ID = 1
# KNICKPOINT_ID = "R06_T_I00452"

THETAS = np.round(np.arange(0.30, 0.701, 0.025), 3)
A0_M2 = 1_000_000.0
MIN_NODOS = 10  # Mínimo numérico exploratorio, no criterio de calidad.


def recortar_perfil(p, inicio, fin):
    x = p["distancia_m"].to_numpy(float)

    if not x[0] <= inicio < fin <= x[-1]:
        raise ValueError("Los límites no forman un intervalo dentro del perfil.")

    interiores = (x > inicio) & (x < fin)

    if interiores.sum() < MIN_NODOS:
        raise ValueError(
            "Hay pocos nodos originales en el intervalo. Revisa los límites."
        )

    xx = np.r_[inicio, x[interiores], fin]
    resultado = pd.DataFrame({"distancia_m": xx})

    for columna in ["elevacion_m", "area_m2", "x_utm", "y_utm"]:
        resultado[columna] = np.interp(xx, x, p[columna])

    return resultado


def ajustar_tramo(tramo, theta):
    x = tramo["distancia_m"].to_numpy(float)
    z = tramo["elevacion_m"].to_numpy(float)
    area = tramo["area_m2"].to_numpy(float)

    chi = cumulative_trapezoid(
        (A0_M2 / area)**theta, x=x, initial=0,
    )

    fit = linregress(chi, z)
    prediccion = fit.intercept + fit.slope * chi

    return {
        "theta": float(theta),
        "M_chi": float(fit.slope),
        "ksn": float(fit.slope * A0_M2**theta),
        "R2": float(fit.rvalue**2),
        "RMSE_m": float(np.sqrt(np.mean((z - prediccion)**2))),
        "chi_final_m": float(chi[-1]),
        "chi": chi,
        "prediccion": prediccion,
    }


# Vaciar resultados para no reutilizar cortes de una ejecución anterior.
tramos = {}
tabla_theta = pd.DataFrame()

if any(v is None for v in [INICIO_ROCA_KM, CRUCE_ID, KNICKPOINT_ID]):
    print("Completa INICIO_ROCA_KM, CRUCE_ID y KNICKPOINT_ID.")
else:
    fila_cruce = cruces[cruces["cruce_id"].eq(CRUCE_ID)]
    fila_kp = knickpoints[
        knickpoints["ID"].eq(RIO_ID)
        & knickpoints["id_kp"].eq(KNICKPOINT_ID)
    ]

    if len(fila_cruce) != 1 or len(fila_kp) != 1:
        raise ValueError("El cruce o el knickpoint seleccionado no existe.")

    x_inicio = float(INICIO_ROCA_KM) * 1000
    x_terraza = float(fila_cruce.iloc[0]["distancia_m"])
    x_kp = float(fila_kp.iloc[0]["distancia_m"])

    if not x_inicio < x_terraza < x_kp:
        raise ValueError(
            "Se requiere: final aluvial < cruce de terraza < knickpoint. "
            "Si no ocurre, esta configuración de dos tramos no es aplicable."
        )

    tramos = {
        "costero": recortar_perfil(p, x_inicio, x_terraza),
        "terraza_knickpoint": recortar_perfil(p, x_terraza, x_kp),
    }

    filas = []

    for nombre, tramo in tramos.items():
        for theta in THETAS:
            ajuste = ajustar_tramo(tramo, theta)
            filas.append({
                "tramo": nombre,
                **{
                    k: v for k, v in ajuste.items()
                    if k not in ["chi", "prediccion"]
                },
            })

    tabla_theta = pd.DataFrame(filas)

    fig, axs = plt.subplots(1, 2, figsize=(13, 4))

    axs[0].plot(
        p["distancia_m"] / 1000, p["elevacion_m"],
        color="0.75", label="Perfil completo",
    )

    for nombre, tramo in tramos.items():
        axs[0].plot(
            tramo["distancia_m"] / 1000, tramo["elevacion_m"],
            lw=2.5, label=nombre,
        )
        datos = tabla_theta[tabla_theta["tramo"].eq(nombre)]
        axs[1].plot(datos["theta"], datos["RMSE_m"], "o-", label=nombre)

        mejor = datos.loc[datos["RMSE_m"].idxmin()]
        extremo = np.isclose(mejor["theta"], [THETAS[0], THETAS[-1]]).any()

        print(
            nombre,
            f": mínimo geométrico en θ={mejor['theta']:.3f}",
            "— en el borde del barrido; no está acotado." if extremo else "",
        )

    axs[0].set(xlabel="Distancia (km)", ylabel="Elevación (m)")
    axs[1].set(xlabel="θ", ylabel="RMSE del ajuste (m)")

    for ax in axs:
        ax.grid(alpha=0.25)
        ax.legend()

    plt.tight_layout()
    plt.show()
    plt.close(fig)


## Escenario de K y tiempo de propagación

Para $n = 1$:

$$
K = \frac{U}{k_{sn}}
$$

$$
\Delta t = \frac{\Delta \chi}{K\,A_0^{\theta}}
$$

$K$ se calibra en el tramo costero suponiendo equilibrio y una tasa
de levantamiento $U$ representativa. Su aplicación entre la terraza
y el knickpoint requiere asumir que $K$ y $\theta$ son transferibles.

$\Delta t$ representa el tiempo modelado de propagación entre los
límites seleccionados; no es automáticamente la edad del knickpoint
o de la falla.

In [ ]:
# ── MODIFICAR AQUÍ DESPUÉS DE REVISAR LOS TRAMOS ───────────────────
THETA_ADOPTADO = 0.45

U_M_KA = None      # Tasa de levantamiento adoptada, en m/ka.
FUENTE_U = ""     # Fuente y justificación de su aplicación a este río.

APLICAR_EQUILIBRIO_COSTERO = False
TRANSFERIR_K_HASTA_KP = False

# Estos interruptores explicitan supuestos del escenario.
# No equivalen a una validación científica automática.

resultado_K = None
delta_t_ka = None

if not tramos:
    print("Primero define los cortes en la celda anterior.")

elif U_M_KA is None:
    print("Introduce U_M_KA y su fuente cuando hayas revisado su aplicabilidad.")

elif not np.isfinite(U_M_KA) or U_M_KA <= 0:
    raise ValueError("U_M_KA debe ser positiva y finita.")

elif not FUENTE_U.strip():
    print("Documenta FUENTE_U antes de calcular K.")

elif not APLICAR_EQUILIBRIO_COSTERO:
    print("K pendiente: falta adoptar y justificar el escenario de equilibrio.")

else:
    ajuste = ajustar_tramo(tramos["costero"], THETA_ADOPTADO)

    if ajuste["ksn"] <= 0:
        raise ValueError("La pendiente ajustada no permite calcular un K positivo.")

    # Convertir m/ka a m/año.
    U_m_anio = float(U_M_KA) / 1000
    K = U_m_anio / ajuste["ksn"]

    resultado_K = {
        "rio": nombres[RIO_ID],
        "terraza": TERRAZA_ID,
        "knickpoint": KNICKPOINT_ID,
        "theta": THETA_ADOPTADO,
        "n": 1,
        "U_m_ka": U_M_KA,
        "ksn_costero": ajuste["ksn"],
        "K": K,
        "unidad_K": f"m^{1 - 2 * THETA_ADOPTADO:g}/año",
        "R2_costero": ajuste["R2"],
        "fuente_U": FUENTE_U,
    }

    display(pd.DataFrame([resultado_K]))

    if TRANSFERIR_K_HASTA_KP:
        superior = ajustar_tramo(
            tramos["terraza_knickpoint"], THETA_ADOPTADO
        )

        delta_t_anios = (
            superior["chi_final_m"]
            / (K * A0_M2**THETA_ADOPTADO)
        )
        delta_t_ka = delta_t_anios / 1000

        print(
            "Tiempo modelado desde el cruce de la terraza hasta el knickpoint:",
            f"{delta_t_ka:.2f} ka",
        )
        print(
            "Escenario con K constante extrapolado desde el tramo costero; "
            "no incluye propagación desde la desembocadura."
        )
    else:
        print("Δt pendiente: no se ha adoptado la transferencia de K.")